# Day 91: Cache-Aware Routing

**Layer:** Advanced


## Concept Overview

Cache-aware routing directs requests to the inference worker that already has the longest matching KV cache prefix. This maximizes KV cache hit rate and avoids recomputing shared prefixes like system prompts.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Prefix Hash Routing

Hash the request prefix at configurable granularity and route to the worker with the longest cache hit.


In [ ]:
import hashlib
import numpy as np

class PrefixRouter:
    def __init__(self, n_workers=4, block_size=16):
        self.n_workers = n_workers
        self.block_size = block_size
        self.caches = [{} for _ in range(n_workers)]  # worker -> {prefix_hash: True}

    def _prefix_hash(self, tokens, n_blocks):
        """Hash the first n_blocks * block_size tokens."""
        prefix = tuple(tokens[:n_blocks * self.block_size])
        return hashlib.md5(str(prefix).encode()).hexdigest()

    def route(self, tokens):
        n_blocks = len(tokens) // self.block_size
        best_worker, best_hit = 0, -1
        for w in range(self.n_workers):
            # Find longest cached prefix
            for b in range(n_blocks, 0, -1):
                h = self._prefix_hash(tokens, b)
                if h in self.caches[w]:
                    if b > best_hit:
                        best_hit = b; best_worker = w
                    break
        # Cache this prefix on routed worker
        for b in range(1, n_blocks+1):
            self.caches[best_worker][self._prefix_hash(tokens, b)] = True
        return best_worker, best_hit

np.random.seed(42)
router = PrefixRouter(n_workers=4)
system_prompt = list(range(64))  # 64 tokens shared prefix
results = []
for i in range(100):
    tokens = system_prompt + list(np.random.randint(0, 1000, size=np.random.randint(20,80)))
    w, hit = router.route(tokens)
    results.append(hit)

hit_rate = sum(1 for h in results if h > 0) / len(results)
print(f"Cache-aware routing: {hit_rate:.0%} hit rate over 100 requests")
print(f"Average blocks cached: {sum(results)/len(results):.1f}")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Cache-aware routing directs requests to the inference worker that already has the longest matching KV cache prefix.
- Cache-aware routing directs requests to the inference worker that already has th....
- Day 91 implementation complete.
